# underfit - Stable Audio 3 LoRA finetuning

**underfit** is a dashboard for LoRA finetunes of Stable Audio 3, provided by [dadabots](https://github.com/dada-bots). 

Sourcecode: https://github.com/dada-bots/underfit

*This notebook is an adaptation of the [Colab notebook](https://github.com/dada-bots/underfit/blob/main/underfit-colab.ipynb), provided in the origial repo. Kaggle specific changes are mentioned below.*

-----

Notebook author: [Martin Heinze](https://github.com/devstermarts)

Last updated: 12.06.2026

## How to run this

While training itself runs on Kaggle, the actual dataset selection/ preparation, finetune configuration and progress checking is done on the underfit dashboard accessible through ngrok.

0. Make sure you have an [ngrok](https://www.ngrok.com) account - free tier is sufficient for starters
1. Make sure you have access to the SA3 model family on [huggingface](https://huggingface.co/stabilityai/stable-audio-3-medium)
2. Setup this notebook on Kaggle (details below). Make sure "Internet on" is toggled in the notebook details.
3. Add your audio dataset via "Add input" section.
4. Select the "GPU T4 x2" option in "Accelerator"
5. "Save version" to run this notebook in the background
6. When you see the "Still running..." message in the logs, go to your ngrok-Dashboard. Your session should pop up in the "Endpoints & Traffic Policy" section <- click.
7. On the underfit dashboard, create a new dataset by using the path to your input dataset, e.g. '/kaggle/input/datasets/my-audio-data' (this can take a while, don't kill the process until it says "Done".
8. Add and configure a new finetune on the main dashboard, set the details -> Launch

---

**Notes:** 
* free tier ngrok has a bandwidth limit of 1GB per month. Use that bandwidth wisely or you'll fly blind at some point :)
* having two T4 GPUs available per run allows you to set up two trainings in parallel, good for parameter testing!
* Kaggle machines run for max 12h before shutting down. If you need to resume training, create a new dataset from your earlier notebook run's output and add it to the notebook before running it again (find a section below, where the earlier run's folder is copied to '/kaggle/working', the rest is done on the underfit dashboard again.
* dadabots' original [Colab notebook](https://github.com/dada-bots/underfit/blob/main/underfit-colab.ipynb) is well documented and comes with additional hints for the training that are not covered here. Highly recommended!

---

## Setup

In [ ]:
import os, shutil, subprocess

gpu_name = None
if shutil.which("nvidia-smi"):
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name",
                            "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=5)
        if r.returncode == 0 and r.stdout.strip():
            gpu_name = r.stdout.strip().splitlines()[0]
    except Exception:
        pass

if gpu_name is None:
    print()
    print("═" * 70)
    print("  ⚠️  NO GPU DETECTED — underfit needs a GPU runtime  ⚠️")
    print("═" * 70)
    print()
    print("  Fix it now:")
    print("    1. Go to 'Session options'")
    print("    2. Select hardware accelerator → GPU T4 x2")
    print("    3. Reload session")
    print("    4. Re-run cell")
    print()
    print("═" * 70)
else:
    print(f"✓ GPU detected: {gpu_name}\n")
    # ── Clone underfit + install deps (uv + venv, ~5 GB of wheels, ~1 min) ──
    if not os.path.isdir("/content/underfit"):
        subprocess.run(["git", "clone",
                        "https://github.com/dada-bots/underfit",
                        "/content/underfit"], check=True)
    os.chdir("/content/underfit")
    subprocess.run(["./install.sh", "--no-setup"], check=True)

## Setup secrets, authenticate huggingface

To run underfit, you need: 
* access to SA3 model family on huggingface
* an ngrok account
* access tokens from both services 

Once you have acquired these, add your secrets via "Add-ons" -> "Secrets" -> Add Secret. 
When done, change below code cell according to the secrets' name. 

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("your-huggingface-token")
secret_value_1 = user_secrets.get_secret("your-ngrok-token")

import os
os.environ['HF_TOKEN'] = secret_value_0

In [ ]:
#Update huggingface_hub and authenticate
!/usr/bin/python3 -m pip install -U huggingface_hub
!hf auth login --token $HF_TOKEN

## Setup paths

In [ ]:
import os

# Persistent user state (runs, datasets, audio) — lives on this notebook's output so it
# survives sessions.
os.environ['UNDERFIT_STATE_DIR'] = '/kaggle/working/underfit-state'

# Model files — stay on local SSD for fast reads. Wiped on session reset, but re-fetched by the wizard.
os.environ['UNDERFIT_MODELS_DIR'] = '/content/models'

# Training logs — write to local SSD for instant dashboard reads. A background
# thread rsyncs to '/kaggle/working' every ~60s for durability.
os.environ['UNDERFIT_LOGS_DIR'] = '/content/underfit-logs'

# State JSON files (runs.json, datasets.json, gradio_*.json). Synced to '/kaggle/working' every ~10s.
os.environ['UNDERFIT_STATE_FILES_DIR'] = '/content/underfit-state'

# Seed-LoRA upload staging. The file gets copied into the run dir (on '/kaggle/working') at launch time.
os.environ['UNDERFIT_SEED_LORAS_DIR'] = '/content/seed_loras'

# HuggingFace cache — also on local SSD.
os.environ['HF_HOME'] = '/content/hf-cache'

import shutil
from pathlib import Path
from huggingface_hub import whoami

_drive_hf_dir = Path('/kaggle/working/hf-cache')
_local_hf_dir = Path(os.environ['HF_HOME'])
_local_hf_dir.mkdir(parents=True, exist_ok=True)

for fname in ('stored_tokens', 'token'):
    src_f = _drive_hf_dir / fname
    dst_f = _local_hf_dir / fname
    if src_f.exists() and not dst_f.exists():
        shutil.copy2(src_f, dst_f)

In [ ]:
#Only when resuming training runs: copy training data from additional Kaggle dataset to '/kaggle/working'
#!cp -r /kaggle/input/datasets/your-earlier-training/* /kaggle/working/

## Download SA3 backend and models

In [ ]:
import os
if not os.path.isdir("/content/stable-audio-3"):
    !git clone https://github.com/Stability-AI/stable-audio-3.git /content/stable-audio-3

!uv run python -m underfit.cli.setup \
    --backend sa3 \
    --backend-path /content/stable-audio-3 \
    --models sa3-medium

#Note: instead of "sa3-medium", you can also pull "sa3-sm-music" or "sa3-sm-sfx"

## Start ngrok dashboard

In [ ]:
import subprocess
from underfit.monitor import launch_dashboard_subprocess, dashboard_button, restart_dashboard_button, start_drive_log_sync, start_drive_state_sync

start_drive_state_sync(
    local_dir='/content/underfit-state',
    drive_dir='/kaggle/working/underfit-state',
    interval=10,
)
start_drive_log_sync(
    local_dir='/content/underfit-logs',
    drive_dir='/kaggle/working/underfit-state/runs',
    interval=60,
)

proc = launch_dashboard_subprocess(port=8787)

import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
from pyngrok import ngrok, conf
conf.get_default().auth_token = secret_value_1
ngrok_url = ngrok.connect(8787, 'http').public_url
dashboard_button(url=ngrok_url, label="🌐 Open Dashboard (via ngrok)")

#Secondary action below the big Open button.
restart_dashboard_button(port=8787)

print("\n(Dashboard is running in the background.)")

In [ ]:
#Poll every 60s to keep alive when committing via 'Save'

import time

while proc.poll() is None:
    time.sleep(60)  # Check every 60 seconds
    print("Still running...")